# 🧠 CNN from Scratch — Fashion MNIST
### UE24CS645BC2 | DLTP Assignment 1

**No PyTorch. No TensorFlow. Pure NumPy.**

This notebook implements:
- ✅ Convolutional Layer (forward + backward)
- ✅ ReLU Activation
- ✅ MaxPooling Layer (forward + backward)
- ✅ Flatten Layer
- ✅ Fully Connected Layer
- ✅ Softmax + Cross-Entropy Loss
- ✅ SGD with Momentum
- ✅ Training & Evaluation on Fashion MNIST

## 📦 Step 1: Install & Import

In [ ]:
import numpy as np
import struct, os, gzip, time, json, urllib.request
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import clear_output

print('NumPy version:', np.__version__)
print('All imports successful ✅')

## 📥 Step 2: Download & Load Fashion MNIST

In [ ]:
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

URLS = {
    'train_images': 'http://fashion-mnist.s3-website.eu-west-1.amazonaws.com/train-images-idx3-ubyte.gz',
    'train_labels': 'http://fashion-mnist.s3-website.eu-west-1.amazonaws.com/train-labels-idx1-ubyte.gz',
    'test_images':  'http://fashion-mnist.s3-website.eu-west-1.amazonaws.com/t10k-images-idx3-ubyte.gz',
    'test_labels':  'http://fashion-mnist.s3-website.eu-west-1.amazonaws.com/t10k-labels-idx1-ubyte.gz',
}

def download_fashion_mnist(data_dir='data'):
    os.makedirs(data_dir, exist_ok=True)
    for name, url in URLS.items():
        path = os.path.join(data_dir, name + '.gz')
        if not os.path.exists(path):
            print(f'Downloading {name}...')
            urllib.request.urlretrieve(url, path)
    print('Download complete ✅')

def load_images(path):
    with gzip.open(path, 'rb') as f:
        _, num, rows, cols = struct.unpack('>IIII', f.read(16))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)

def load_labels(path):
    with gzip.open(path, 'rb') as f:
        _, num = struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8)

def load_fashion_mnist(data_dir='data'):
    download_fashion_mnist(data_dir)
    X_train = load_images(os.path.join(data_dir, 'train_images.gz')).astype(np.float32) / 255.0
    y_train = load_labels(os.path.join(data_dir, 'train_labels.gz'))
    X_test  = load_images(os.path.join(data_dir, 'test_images.gz')).astype(np.float32) / 255.0
    y_test  = load_labels(os.path.join(data_dir, 'test_labels.gz'))
    # Add channel dim: (N,H,W) -> (N,1,H,W)
    X_train = X_train[:, np.newaxis, :, :]
    X_test  = X_test[:, np.newaxis, :, :]
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_fashion_mnist()
print(f'Train images: {X_train.shape}  Labels: {y_train.shape}')
print(f'Test  images: {X_test.shape}   Labels: {y_test.shape}')

## 🖼️ Step 3: Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Fashion MNIST — Sample Images', fontsize=14, fontweight='bold')
for c in range(10):
    ax = axes[c // 5][c % 5]
    idx = np.where(y_train == c)[0][0]
    ax.imshow(X_train[idx, 0], cmap='gray')
    ax.set_title(CLASS_NAMES[c], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 🏗️ Step 4: Build CNN Layers from Scratch

### 4.1 — Convolutional Layer

In [ ]:
class ConvLayer:
    """
    2D Convolutional Layer — built from scratch.

    Forward:
        out[n, f, i, j] = Σ_{c,kh,kw} W[f,c,kh,kw] · X[n,c,i+kh,j+kw] + b[f]

    Backward:
        dW[f]  = Σ_n  patch(X, i, j) · dout[n, f, i, j]   (correlation)
        db[f]  = Σ_{n,i,j} dout[n, f, i, j]
        dX     = full convolution of dout with flipped kernel
    """

    def __init__(self, in_channels, num_filters, kernel_size, stride=1, padding=0):
        self.in_channels  = in_channels
        self.num_filters  = num_filters
        self.kernel_size  = kernel_size
        self.stride       = stride
        self.padding      = padding

        # Xavier initialization — prevents vanishing/exploding gradients
        fan_in  = in_channels * kernel_size * kernel_size
        fan_out = num_filters * kernel_size * kernel_size
        scale   = np.sqrt(2.0 / (fan_in + fan_out))
        self.W  = np.random.randn(num_filters, in_channels, kernel_size, kernel_size).astype(np.float32) * scale
        self.b  = np.zeros(num_filters, dtype=np.float32)

        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self._X_padded = None
        self._X_shape  = None

    def _pad(self, X):
        if self.padding == 0:
            return X
        return np.pad(X, ((0,0),(0,0),(self.padding,self.padding),(self.padding,self.padding)), mode='constant')

    def forward(self, X):
        """X: (N, C, H, W) → out: (N, F, H_out, W_out)"""
        N, C, H, W = X.shape
        F, _, KH, KW = self.W.shape
        s, p = self.stride, self.padding
        H_out = (H + 2*p - KH) // s + 1
        W_out = (W + 2*p - KW) // s + 1

        X_padded = self._pad(X)
        self._X_padded = X_padded
        self._X_shape  = X.shape

        out = np.zeros((N, F, H_out, W_out), dtype=np.float32)
        for f in range(F):
            for i in range(H_out):
                for j in range(W_out):
                    patch = X_padded[:, :, i*s:i*s+KH, j*s:j*s+KW]   # (N,C,KH,KW)
                    out[:, f, i, j] = np.tensordot(patch, self.W[f], axes=([1,2,3],[0,1,2])) + self.b[f]
        return out

    def backward(self, dout):
        """dout: (N, F, H_out, W_out) → dX: (N, C, H, W)"""
        N, C, H, W = self._X_shape
        F, _, KH, KW = self.W.shape
        s, p = self.stride, self.padding
        H_out, W_out = dout.shape[2], dout.shape[3]

        X_padded  = self._X_padded
        dX_padded = np.zeros_like(X_padded)
        self.dW   = np.zeros_like(self.W)
        self.db   = np.zeros_like(self.b)

        for f in range(F):
            self.db[f] = np.sum(dout[:, f, :, :])
            for i in range(H_out):
                for j in range(W_out):
                    patch = X_padded[:, :, i*s:i*s+KH, j*s:j*s+KW]   # (N,C,KH,KW)
                    d     = dout[:, f, i, j]                           # (N,)
                    self.dW[f]   += np.tensordot(d, patch, axes=([0],[0]))
                    dX_padded[:, :, i*s:i*s+KH, j*s:j*s+KW] += (
                        self.W[f][np.newaxis] * d[:, np.newaxis, np.newaxis, np.newaxis]
                    )

        if p > 0:
            return dX_padded[:, :, p:-p, p:-p]
        return dX_padded

print('ConvLayer defined ✅')

### 4.2 — ReLU Activation

In [ ]:
class ReLU:
    """
    Rectified Linear Unit: f(x) = max(0, x)
    Backward: gradient passes through where x > 0, else 0.
    """
    def __init__(self):  self._mask = None
    def forward(self, X):
        self._mask = (X > 0)
        return np.maximum(0, X)
    def backward(self, dout):
        return dout * self._mask

print('ReLU defined ✅')

### 4.3 — MaxPooling Layer

In [ ]:
class MaxPoolLayer:
    """
    Max Pooling Layer.

    Forward:
        Slides a (pool_size × pool_size) window over each feature map,
        retaining only the maximum value → reduces spatial dimensions.

    Backward:
        Gradient flows only to the position that held the maximum value.
        All other positions receive zero gradient (max routing).
    """

    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride    = stride
        self._X        = None
        self._max_mask = None

    def forward(self, X):
        """X: (N, C, H, W) → (N, C, H_out, W_out)"""
        N, C, H, W = X.shape
        ps, s = self.pool_size, self.stride
        H_out = (H - ps) // s + 1
        W_out = (W - ps) // s + 1

        self._X        = X
        self._max_mask = np.zeros_like(X, dtype=bool)
        out = np.zeros((N, C, H_out, W_out), dtype=np.float32)

        for i in range(H_out):
            for j in range(W_out):
                h0, w0 = i*s, j*s
                patch  = X[:, :, h0:h0+ps, w0:w0+ps]                   # (N,C,ps,ps)
                max_v  = np.max(patch, axis=(2,3), keepdims=True)        # (N,C,1,1)
                out[:, :, i, j] = max_v[:, :, 0, 0]
                self._max_mask[:, :, h0:h0+ps, w0:w0+ps] |= (patch == max_v)

        return out

    def backward(self, dout):
        """dout: (N, C, H_out, W_out) → dX: (N, C, H, W)"""
        ps, s = self.pool_size, self.stride
        H_out, W_out = dout.shape[2], dout.shape[3]
        dX = np.zeros_like(self._X)

        for i in range(H_out):
            for j in range(W_out):
                h0, w0  = i*s, j*s
                mask    = self._max_mask[:, :, h0:h0+ps, w0:w0+ps]
                d       = dout[:, :, i, j][:, :, np.newaxis, np.newaxis]
                count   = mask.sum(axis=(2,3), keepdims=True).clip(min=1)
                dX[:, :, h0:h0+ps, w0:w0+ps] += mask * d / count

        return dX

print('MaxPoolLayer defined ✅')

### 4.4 — Flatten Layer

In [ ]:
class FlattenLayer:
    """Reshapes (N, C, H, W) → (N, C*H*W) for the FC layers."""
    def __init__(self):  self._shape = None
    def forward(self, X):
        self._shape = X.shape
        return X.reshape(X.shape[0], -1)
    def backward(self, dout):
        return dout.reshape(self._shape)

print('FlattenLayer defined ✅')

### 4.5 — Fully Connected Layer

In [ ]:
class FCLayer:
    """
    Fully Connected (Dense) Layer.

    Forward:  out = X @ W + b
    Backward: dW  = Xᵀ @ dout
              db  = sum(dout, axis=0)
              dX  = dout @ Wᵀ
    """
    def __init__(self, in_features, out_features):
        scale   = np.sqrt(2.0 / in_features)    # He initialization
        self.W  = np.random.randn(in_features, out_features).astype(np.float32) * scale
        self.b  = np.zeros(out_features, dtype=np.float32)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self._X = None

    def forward(self, X):
        self._X = X
        return X @ self.W + self.b

    def backward(self, dout):
        self.dW = self._X.T @ dout
        self.db = dout.sum(axis=0)
        return dout @ self.W.T

print('FCLayer defined ✅')

### 4.6 — Softmax + Cross-Entropy Loss

In [ ]:
def softmax(logits):
    """Numerically stable softmax: subtract max before exp."""
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_x   = np.exp(shifted)
    return exp_x / exp_x.sum(axis=1, keepdims=True)

def cross_entropy_loss(probs, y):
    """
    Cross-Entropy Loss + gradient w.r.t. logits.

    The combined softmax + cross-entropy gradient simplifies to:
        ∂L/∂logits = (probs - one_hot(y)) / N

    This is much simpler and more stable than computing them separately.
    """
    N   = probs.shape[0]
    eps = 1e-9
    loss = -np.log(probs[np.arange(N), y] + eps).mean()

    dlogits          = probs.copy()
    dlogits[np.arange(N), y] -= 1
    dlogits         /= N
    return loss, dlogits

print('Softmax + CrossEntropy defined ✅')

## 🔗 Step 5: Assemble the CNN Model

In [ ]:
class CNN:
    """
    CNN Architecture:

        Input (N,1,28,28)
        → Conv(1→8, 3×3, pad=1) → ReLU → MaxPool(2×2)   # (N, 8,14,14)
        → Conv(8→16,3×3, pad=1) → ReLU → MaxPool(2×2)   # (N,16, 7, 7)
        → Flatten                                          # (N, 784)
        → FC(784→128) → ReLU
        → FC(128→10)
        → Softmax + CrossEntropy
    """

    def __init__(self):
        self.conv1   = ConvLayer(1,  8,  3, padding=1)
        self.relu1   = ReLU()
        self.pool1   = MaxPoolLayer(2, 2)

        self.conv2   = ConvLayer(8,  16, 3, padding=1)
        self.relu2   = ReLU()
        self.pool2   = MaxPoolLayer(2, 2)

        self.flatten = FlattenLayer()

        self.fc1     = FCLayer(16 * 7 * 7, 128)
        self.relu3   = ReLU()
        self.fc2     = FCLayer(128, 10)

        self.layers  = [
            self.conv1, self.relu1, self.pool1,
            self.conv2, self.relu2, self.pool2,
            self.flatten,
            self.fc1, self.relu3,
            self.fc2
        ]

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out   # logits

    def backward(self, dlogits):
        grad = dlogits
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def predict(self, X):
        return np.argmax(softmax(self.forward(X)), axis=1)

    def get_params_and_grads(self):
        for layer in [self.conv1, self.conv2, self.fc1, self.fc2]:
            yield layer.W, layer.dW
            yield layer.b, layer.db

    def summary(self):
        print('=' * 45)
        print(f'{"CNN Architecture":^45}')
        print('=' * 45)
        rows = [
            ('Input',          '(N, 1, 28, 28)'),
            ('Conv2D(8, 3×3)', '(N, 8, 28, 28)'),
            ('ReLU',           '(N, 8, 28, 28)'),
            ('MaxPool(2×2)',    '(N, 8, 14, 14)'),
            ('Conv2D(16,3×3)', '(N,16, 28, 28)'),
            ('ReLU',           '(N,16, 14, 14)'),
            ('MaxPool(2×2)',    '(N,16,  7,  7)'),
            ('Flatten',        '(N, 784)'),
            ('FC(784→128)',     '(N, 128)'),
            ('ReLU',           '(N, 128)'),
            ('FC(128→10)',      '(N, 10)'),
            ('Softmax',        '(N, 10)'),
        ]
        for name, shape in rows:
            print(f'  {name:<20}  →  {shape}')
        print('=' * 45)

model = CNN()
model.summary()

## ⚡ Step 6: Optimizer — SGD with Momentum

In [ ]:
class SGDMomentum:
    """
    SGD with Momentum:
        v  = μ·v  - lr·∇
        θ += v

    Momentum (μ=0.9) helps accelerate gradients in consistent directions
    and dampens oscillations, leading to faster and smoother convergence.
    """
    def __init__(self, model, lr=0.01, momentum=0.9):
        self.lr       = lr
        self.momentum = momentum
        self.velocity = {id(p): np.zeros_like(p)
                         for p, _ in model.get_params_and_grads()}

    def step(self, model):
        for param, grad in model.get_params_and_grads():
            v = self.velocity[id(param)]
            v[:] = self.momentum * v - self.lr * grad
            param += v

optimizer = SGDMomentum(model, lr=0.01, momentum=0.9)
print('Optimizer ready ✅')

## 🏋️ Step 7: Training & Evaluation Functions

In [ ]:
def evaluate(model, X, y, batch_size=256):
    """Compute accuracy on a dataset in mini-batches."""
    correct = 0
    for start in range(0, len(X), batch_size):
        Xb = X[start:start+batch_size]
        yb = y[start:start+batch_size]
        correct += (model.predict(Xb) == yb).sum()
    return correct / len(X)


def train(model, optimizer, X_train, y_train, X_test, y_test,
          epochs=10, batch_size=64):
    """
    Mini-batch Training Loop:
      1. Shuffle training data each epoch
      2. For each mini-batch:
         a. Forward pass → logits
         b. Softmax → probabilities
         c. Cross-entropy loss + gradient
         d. Backward pass (backpropagation)
         e. Update parameters via optimizer
      3. Evaluate on test set after each epoch
    """
    N       = len(X_train)
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    print(f'{'─'*65}')
    print(f'{"Epoch":>6}  {"Train Loss":>12}  {"Train Acc":>10}  {"Test Acc":>10}  {"Time":>6}')
    print(f'{'─'*65}')

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        idx = np.random.permutation(N)
        X_s, y_s = X_train[idx], y_train[idx]

        epoch_loss = 0.0
        correct    = 0

        for start in range(0, N, batch_size):
            Xb, yb = X_s[start:start+batch_size], y_s[start:start+batch_size]

            # ── Forward ──
            logits = model.forward(Xb)
            probs  = softmax(logits)

            # ── Loss ──
            loss, dlogits = cross_entropy_loss(probs, yb)
            epoch_loss   += loss * len(Xb)
            correct      += (np.argmax(probs, 1) == yb).sum()

            # ── Backward ──
            model.backward(dlogits)

            # ── Update ──
            optimizer.step(model)

        train_loss = epoch_loss / N
        train_acc  = correct   / N
        test_acc   = evaluate(model, X_test, y_test)

        history['train_loss'].append(float(train_loss))
        history['train_acc'].append(float(train_acc))
        history['test_acc'].append(float(test_acc))

        elapsed = time.time() - t0
        print(f'{epoch:>6}  {train_loss:>12.4f}  {train_acc*100:>9.2f}%  {test_acc*100:>9.2f}%  {elapsed:>5.1f}s')

    print(f'{'─'*65}')
    return history

print('Train & evaluate functions ready ✅')

## 🚀 Step 8: Run Training

> **💡 Tip:** Change `SUBSET = 10000` to `SUBSET = None` to train on the full 60k dataset (takes ~30 min on CPU).

In [ ]:
np.random.seed(42)

# ── Subset for faster demo (set None for full training) ──
SUBSET     = 10000
EPOCHS     = 10
BATCH_SIZE = 64
LR         = 0.01

if SUBSET:
    idx = np.random.choice(len(X_train), SUBSET, replace=False)
    X_tr, y_tr = X_train[idx], y_train[idx]
    print(f'Using {SUBSET} training samples')
else:
    X_tr, y_tr = X_train, y_train
    print('Using full 60,000 training samples')

# ── Fresh model + optimizer ──
model     = CNN()
optimizer = SGDMomentum(model, lr=LR, momentum=0.9)

print(f'Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | LR: {LR}\n')
history = train(model, optimizer, X_tr, y_tr, X_test, y_test,
                epochs=EPOCHS, batch_size=BATCH_SIZE)

## 📊 Step 9: Plot Training Curves

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('CNN Training History', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history['train_loss'], 'o-', color='#e74c3c', label='Train Loss')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'o-', color='#2ecc71', label='Train Acc')
axes[1].plot(epochs_range, [a*100 for a in history['test_acc']],  's--', color='#3498db', label='Test Acc')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 📋 Step 10: Detailed Per-Class Evaluation

In [ ]:
# ── Collect all predictions ──
all_preds = []
for start in range(0, len(X_test), 256):
    all_preds.extend(model.predict(X_test[start:start+256]))
all_preds = np.array(all_preds)

overall_acc = (all_preds == y_test).mean()

print(f'\n{'='*52}')
print(f'  Overall Test Accuracy: {overall_acc*100:.2f}%')
print(f'{'='*52}')
print(f'{'Class':<20} {"Correct":>8} {"Total":>7} {"Acc":>8}')
print('-'*52)
per_class_acc = []
for c in range(10):
    mask = y_test == c
    corr = (all_preds[mask] == c).sum()
    tot  = mask.sum()
    acc  = corr / tot * 100
    per_class_acc.append(acc)
    print(f'{CLASS_NAMES[c]:<20} {corr:>8} {tot:>7} {acc:>7.1f}%')
print('='*52)

## 📊 Step 11: Per-Class Accuracy Bar Chart

In [ ]:
colors = ['#e74c3c' if a < 70 else '#f39c12' if a < 80 else '#2ecc71'
          for a in per_class_acc]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(CLASS_NAMES, per_class_acc, color=colors, edgecolor='white', linewidth=0.5)

for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.axhline(overall_acc * 100, color='#2c3e50', linestyle='--', linewidth=1.5,
           label=f'Overall: {overall_acc*100:.1f}%')
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Test Accuracy — CNN from Scratch', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved per_class_accuracy.png')

## 🔍 Step 12: Visualize Predictions

In [ ]:
np.random.seed(7)
sample_idx = np.random.choice(len(X_test), 20, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(14, 11))
fig.suptitle('CNN Predictions on Fashion MNIST Test Set', fontsize=13, fontweight='bold')

for ax, idx in zip(axes.flat, sample_idx):
    img   = X_test[idx, 0]
    true  = y_test[idx]
    pred  = all_preds[idx]
    color = '#2ecc71' if pred == true else '#e74c3c'

    ax.imshow(img, cmap='gray')
    ax.set_title(f'True: {CLASS_NAMES[true]}\nPred: {CLASS_NAMES[pred]}',
                 fontsize=8, color=color, fontweight='bold')
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)
        spine.set_visible(True)

plt.tight_layout()
plt.savefig('predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Green border = correct  |  Red border = wrong')

## 🔬 Step 13: Visualize Learned Conv Filters

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('Learned Filters — Conv Layer 1 (top) & Conv Layer 2 (bottom, first 8)', fontsize=11, fontweight='bold')

for i in range(8):
    f1 = model.conv1.W[i, 0]   # (3,3)
    axes[0, i].imshow(f1, cmap='RdBu_r', vmin=-f1.abs_max if hasattr(f1,'abs_max') else -np.abs(f1).max(), vmax=np.abs(f1).max())
    axes[0, i].set_title(f'F1-{i+1}', fontsize=8)
    axes[0, i].axis('off')

for i in range(8):
    f2 = model.conv2.W[i, 0]   # (3,3)
    axes[1, i].imshow(f2, cmap='RdBu_r', vmin=-np.abs(f2).max(), vmax=np.abs(f2).max())
    axes[1, i].set_title(f'F2-{i+1}', fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('filters.png', dpi=120, bbox_inches='tight')
plt.show()

## 💾 Step 14: Save Results

In [ ]:
results = {
    'overall_test_accuracy': float(overall_acc),
    'per_class_accuracy': {CLASS_NAMES[c]: float(per_class_acc[c]) for c in range(10)},
    'history': history,
    'hyperparameters': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LR,
        'momentum': 0.9,
        'subset': SUBSET
    }
}

with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to results.json')
print(f'\nFinal Test Accuracy: {overall_acc*100:.2f}%')
print('\nDone! 🎉')